# Peer University Data Check

Load and inspect Scopus exports from Lehigh, Villanova, and Marquette.
Compare against AUB's Scopus export to confirm column compatibility before combining.

## Expected file locations
```
data/raw/scopus.csv              ← AUB (existing)
data/raw/lehigh_scopus.csv       ← Lehigh
data/raw/villanova_scopus.csv    ← Villanova
data/raw/marquette_scopus.csv    ← Marquette
```

In [ ]:
import sys
sys.path.append('../')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
plt.style.use('seaborn-v0_8')
%matplotlib inline

## 1. File paths — update these if your filenames differ

In [ ]:
FILES = {
    'AUB':       '../data/raw/aub_merged_data.csv',
    'Lehigh':    '../data/raw/lehigh.csv',
    'Villanova': '../data/raw/villanova.csv',
    'Marquette': '../data/raw/marquette.csv',
}

# Verify which files exist
for inst, path in FILES.items():
    exists = Path(path).exists()
    status = '✓' if exists else '✗ MISSING'
    print(f"{status}  {inst}: {path}")

## 2. Load all files

In [ ]:
dfs = {}

for inst, path in FILES.items():
    p = Path(path)
    if not p.exists():
        print(f"SKIPPING {inst} — file not found: {path}")
        continue
    try:
        df = pd.read_csv(path, low_memory=False)
        dfs[inst] = df
        print(f"{inst}: {df.shape[0]:,} rows × {df.shape[1]} columns")
    except Exception as e:
        print(f"ERROR loading {inst}: {e}")

print(f"\nLoaded {len(dfs)} / {len(FILES)} institutions")

## 3. Column inventory — what columns does each institution have?

In [ ]:
# Build a column presence matrix
all_cols = sorted(set(col for df in dfs.values() for col in df.columns))
col_matrix = pd.DataFrame(
    {inst: [col in df.columns for col in all_cols] for inst, df in dfs.items()},
    index=all_cols
)

col_matrix['in_all'] = col_matrix.all(axis=1)
col_matrix['count']  = col_matrix.drop(columns='in_all').sum(axis=1)

print(f"Total unique columns across all institutions: {len(all_cols)}")
print(f"Columns present in ALL institutions: {col_matrix['in_all'].sum()}")
print()
col_matrix.sort_values('count', ascending=False)

In [ ]:
# Columns missing from at least one institution
print("Columns NOT shared across all institutions:")
col_matrix[~col_matrix['in_all']].drop(columns='in_all').sort_values('count', ascending=False)

## 4. Key feature check

In [ ]:
# Features we need for the pipeline
KEY_FEATURES = [
    'EID', 'Title', 'Abstract', 'Authors',
    'Citations',         # citation count (SciVal column name)
    'Year', 'Source title', 'Document Type',
    'Author Keywords', 'Index Keywords',
    'DOI', 'Publisher', 'Affiliations',
]

print(f"{'Feature':<25} " + "  ".join(f"{inst:<12}" for inst in dfs))
print("-" * (25 + 14 * len(dfs)))

for feat in KEY_FEATURES:
    row = f"{feat:<25} "
    for inst, df in dfs.items():
        if feat in df.columns:
            row += f"  {'✓':<12}"
        else:
            close = [c for c in df.columns if feat.lower() in c.lower()]
            row += f"  {('~' + close[0])[:12]:<12}" if close else f"  {'✗':<12}"
    print(row)

## 5. Citation column — locate and compare

In [ ]:
citation_keywords = ['citation', 'cited', 'cit']

for inst, df in dfs.items():
    matches = [c for c in df.columns if any(kw in c.lower() for kw in citation_keywords)]
    print(f"{inst}: {matches if matches else 'NO citation column found'}")

In [ ]:
CITATION_COL = 'Citations'

fig, axes = plt.subplots(1, len(dfs), figsize=(5 * len(dfs), 4), sharey=False)
if len(dfs) == 1:
    axes = [axes]

for ax, (inst, df) in zip(axes, dfs.items()):
    if CITATION_COL in df.columns:
        vals = pd.to_numeric(df[CITATION_COL], errors='coerce').dropna()
        ax.hist(vals.clip(upper=vals.quantile(0.99)), bins=50, edgecolor='black')
        ax.set_title(f"{inst}\nmedian={vals.median():.0f}, n={len(vals):,}")
        ax.set_xlabel('Citations')
    else:
        ax.text(0.5, 0.5, f"'{CITATION_COL}'\nnot found", ha='center', va='center',
                transform=ax.transAxes, fontsize=12, color='red')
        ax.set_title(inst)

plt.suptitle('Citation Count Distribution by Institution (clipped at 99th pct)', y=1.02)
plt.tight_layout()
plt.show()

## 6. Missing values per institution

In [ ]:
for inst, df in dfs.items():
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(1)
    report = pd.DataFrame({'Missing': missing, 'Pct': missing_pct})
    report = report[report['Missing'] > 0].sort_values('Missing', ascending=False)

    print(f"\n{'='*50}")
    print(f"{inst} — {len(df):,} rows")
    print(f"{'='*50}")
    if report.empty:
        print("No missing values!")
    else:
        print(report.head(20).to_string())

## 7. EID check — format consistency and cross-institution overlap

In [ ]:
eid_sets = {}

for inst, df in dfs.items():
    if 'EID' not in df.columns:
        print(f"{inst}: NO EID column")
        continue

    eids = df['EID'].dropna().astype(str)
    eid_sets[inst] = set(eids)

    dups = df['EID'].duplicated().sum()
    sample = eids.head(3).tolist()
    print(f"{inst}: {len(eids):,} EIDs | {dups} duplicates | sample: {sample}")

# Cross-institution overlap
if len(eid_sets) > 1:
    print("\nCross-institution EID overlap:")
    insts = list(eid_sets.keys())
    for i in range(len(insts)):
        for j in range(i+1, len(insts)):
            a, b = insts[i], insts[j]
            overlap = len(eid_sets[a] & eid_sets[b])
            print(f"  {a} ∩ {b}: {overlap:,} shared EIDs")

## 8. Abstract quality check

In [ ]:
for inst, df in dfs.items():
    if 'Abstract' not in df.columns:
        print(f"{inst}: No Abstract column")
        continue

    total = len(df)
    missing = df['Abstract'].isna().sum()
    present = total - missing

    lengths = df['Abstract'].dropna().str.len()
    # Flag likely placeholder/URL abstracts (very short or starts with http)
    placeholder = df['Abstract'].dropna().apply(
        lambda x: len(str(x)) < 50 or str(x).strip().startswith('http')
    ).sum()

    print(f"\n{inst}:")
    print(f"  Present: {present:,} / {total:,} ({present/total*100:.1f}%)")
    print(f"  Placeholders/URLs: {placeholder:,}")
    print(f"  Median length: {lengths.median():.0f} chars")
    print(f"  Min length: {lengths.min():.0f} chars")

## 9. Year distribution

In [ ]:
year_col_candidates = ['Year', 'Publication Year', 'year']

fig, axes = plt.subplots(1, len(dfs), figsize=(5 * len(dfs), 4), sharey=False)
if len(dfs) == 1:
    axes = [axes]

for ax, (inst, df) in zip(axes, dfs.items()):
    year_col = next((c for c in year_col_candidates if c in df.columns), None)
    if year_col:
        years = pd.to_numeric(df[year_col], errors='coerce').dropna()
        years.value_counts().sort_index().plot(kind='bar', ax=ax, width=0.8)
        ax.set_title(f"{inst} (col='{year_col}')")
        ax.set_xlabel('Year')
        ax.tick_params(axis='x', rotation=45)
    else:
        ax.text(0.5, 0.5, 'No year column found', ha='center', va='center',
                transform=ax.transAxes, color='red')
        ax.set_title(inst)

plt.suptitle('Publication Year Distribution', y=1.02)
plt.tight_layout()
plt.show()

## 10. Summary table

In [ ]:
rows = []
for inst, df in dfs.items():
    year_col = next((c for c in ['Year', 'Publication Year'] if c in df.columns), None)
    cit_col  = next((c for c in df.columns if 'citation' in c.lower() or c == 'Citations'), None)

    row = {
        'Institution': inst,
        'Rows': len(df),
        'Columns': len(df.columns),
        'Has EID': 'EID' in df.columns,
        'Has Abstract': 'Abstract' in df.columns,
        'Citation col': cit_col or 'MISSING',
        'Year col': year_col or 'MISSING',
        'Missing Abstract %': f"{df['Abstract'].isna().mean()*100:.1f}%" if 'Abstract' in df.columns else 'N/A',
        'Missing Citation %': f"{pd.to_numeric(df[cit_col], errors='coerce').isna().mean()*100:.1f}%" if cit_col else 'N/A',
    }
    rows.append(row)

summary = pd.DataFrame(rows).set_index('Institution')
summary

## Notes / Action Items

Fill in after running:
- [ ] Confirm citation column is `Citations` across all institutions
- [ ] Note any institutions missing Abstract or EID
- [ ] Record cross-institution EID overlap count
- [ ] Flag any year range anomalies
- [ ] Next step → `05_peer_uni_merge.ipynb`